# SparK: 복원 결과 시각화 (tif 패치용)
`make_tif_patches.py`로 만든 **[0, 1] float tif 패치**로 사전학습한 SparK 모델이 마스킹된 영역을 얼마나 복원하는지 확인하는 노트북입니다.

- 입력 전처리는 학습과 같습니다: tif를 그대로 읽고(정규화 없음), `--tif_bands`로 밴드를 고르고, 1밴드는 3채널로 복제합니다.
- 체크포인트는 `main.py`가 `--exp_dir`에 저장하는 `<model>_withdecoder_1kpretrained_spark_style.pth`를 사용합니다. (epoch이 끝날 때마다 덮어써지므로, 비교하려면 복사해 두세요.)
- 이 노트북은 `SparK/pretrain/` 폴더에서 실행하세요.

**무엇을 보면 되나**
1. *복원 영상*: 마스크된 영역에서 건물·구조물의 윤곽이 복원되는가, 아니면 평균 밝기로만 채워지는가.
2. *수치*: 학습 loss와 같은 방식(마스크 영역, 패치별 정규화)의 loss를 여러 패치에서 계산해, 단순 기준선과 비교합니다.

## 1. 설정

In [ ]:
import os, sys, random
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, os.getcwd())     # run this notebook under SparK/pretrain/

# ===================== 여기를 수정하세요 =====================
CKPT = './exp/resnet101vd_withdecoder_1kpretrained_spark_style.pth'   # --exp_dir 안의 체크포인트
PATCH_DIR = '/path/to/patches'      # 학습에 쓴 --data_path (make_tif_patches.py의 --dst_dir)
TIF_BANDS = ''                      # 학습 때 쓴 --tif_bands 와 같게 (예: '1' 또는 '1,2,3'; ''는 모든 밴드)
SCENE_FILTER = ''                   # 비우면 전체에서 무작위 선택; 값을 주면 경로에 이 문자열이 포함된 패치만 사용
N_SHOW = 6                          # 그림으로 볼 패치 수
N_EVAL = 64                         # 수치 평가에 쓸 패치 수
SEED = 0
MASK_RATIO = None                   # None이면 체크포인트(학습) 때의 마스크 비율을 사용
DISPLAY_GAMMA = 0.5                 # 표시용 감마(SAR처럼 어두운 영상을 보기 쉽게); 1.0이면 그대로 표시
USING_GPU_IF_AVAILABLE = True
# =============================================================

_ = torch.empty(1)
if torch.cuda.is_available() and USING_GPU_IF_AVAILABLE:
    _ = _.cuda()
DEVICE = _.device
print(f'[DEVICE={DEVICE}]')

## 2. SparK 모델 불러오기

In [ ]:
from decoder import LightDecoder
from models import build_sparse_encoder
from spark import SparK

def build_spark(ckpt: str, mask_ratio=None):
    assert os.path.exists(ckpt), f'checkpoint not found: {ckpt}'
    try:
        all_state = torch.load(ckpt, map_location='cpu', weights_only=False)
    except TypeError:   # torch < 1.13 has no `weights_only`
        all_state = torch.load(ckpt, map_location='cpu')
    input_size, arch, state = all_state['input_size'], all_state['arch'], all_state['module']
    config = state['config']
    print(f"[ckpt] arch={arch}, input_size={input_size}, epoch(0-based)={all_state.get('epoch')}, "
          f"min/last recon loss={all_state.get('performance_desc')}")
    
    enc = build_sparse_encoder(arch, input_size=input_size, sbn=False)
    spark = SparK(
        sparse_encoder=enc, dense_decoder=LightDecoder(enc.downsample_raito, sbn=False),
        mask_ratio=mask_ratio or config['mask_ratio'], densify_norm=config['densify_norm_str'], sbn=False,
    ).to(DEVICE)
    spark.eval(), [p.requires_grad_(False) for p in spark.parameters()]
    # single-process inference uses plain BatchNorm: SyncBatchNorm has the same weights, so only the flag in the config differs
    state['config'] = {**config, 'sbn': False, 'mask_ratio': spark.mask_ratio}
    missing, unexpected = spark.load_state_dict(state, strict=False)
    assert len(missing) == 0, f'load_state_dict missing keys: {missing}'
    assert len(unexpected) == 0, f'load_state_dict unexpected keys: {unexpected}'
    print(f'[SparK] mask_ratio={spark.mask_ratio}, downsample ratio={spark.downsample_raito}, feature map={spark.fmap_h}x{spark.fmap_w}')
    return spark, input_size

spark, input_size = build_spark(CKPT, MASK_RATIO)

## 3. 패치 불러오기
학습과 같은 방식으로 읽습니다(`utils/tif_dataset.py`). 패치 목록 캐시(`.spark_cache`)가 있으면 바로 사용합니다.
패치가 `input_size`보다 크면 가운데를 잘라서 씁니다(축소하지 않습니다).

In [ ]:
from utils.tif_dataset import TifPatchDataset

bands = [int(b) for b in TIF_BANDS.replace(' ', '').split(',') if b] or None
dataset = TifPatchDataset(PATCH_DIR, transform=lambda x: x, bands=bands)
print(f'{len(dataset):,} patches in {PATCH_DIR}')

def center_crop(x_chw: torch.Tensor, size: int):
    h, w = x_chw.shape[-2:]
    assert h >= size and w >= size, f'patch {h}x{w} is smaller than input_size={size}'
    top, left = (h - size) // 2, (w - size) // 2
    return x_chw[:, top:top + size, left:left + size]

def pick_patches(n: int, seed: int):
    rng = random.Random(seed)
    if not SCENE_FILTER:
        idx = rng.sample(range(len(dataset)), min(n, len(dataset)))
    else:
        idx, tries = [], 0
        while len(idx) < n and tries < 200000:
            i = rng.randrange(len(dataset)); tries += 1
            if SCENE_FILTER in dataset.get_path(i) and i not in idx:
                idx.append(i)
        assert idx, f'no patch whose path contains {SCENE_FILTER!r}'
    return idx

def load_batch(idx):
    return torch.stack([center_crop(dataset[i], input_size) for i in idx])    # (B, 3, S, S), values in [0, 1]

## 4. 시각화 함수
- 마스킹: SparK와 같은 방식(32×32 픽셀 단위 격자에서 무작위로 가림).
- 오른쪽 끝 그림은 **복원 오차**(가려진 영역에서 |복원 − 원본|)입니다. 밝을수록 틀린 곳입니다.
- `loss`는 학습 loss와 같은 방식으로 계산한 값입니다(가려진 패치별로 정규화한 정답과의 오차; 0만 내놓으면 1.0).

In [ ]:
@torch.no_grad()
def masked_loss(spark: SparK, inp_bchw, rec_bchw, active_b1ff):
    """ the reconstruction loss of `SparK.forward` for each sample: (B,) """
    p_inp, p_rec = spark.patchify(inp_bchw), spark.patchify(rec_bchw)    # (B, L, N)
    var = (p_inp.var(dim=-1, keepdim=True) + 1e-6) ** .5                 # same normalization as the training loss
    l2 = (((p_rec - p_inp) / var) ** 2).mean(dim=-1)                     # (B, L)
    non_active = active_b1ff.logical_not().float().view(active_b1ff.shape[0], -1)
    return (l2 * non_active).sum(dim=1) / non_active.sum(dim=1).clamp(min=1)

def to_img(chw: torch.Tensor):
    """ (3, H, W) in [0, 1] => numpy image for imshow (gray if all channels are the same) """
    x = chw.detach().float().clamp(0., 1.).pow(DISPLAY_GAMMA).cpu()
    if torch.allclose(x[0], x[1]) and torch.allclose(x[0], x[2]):
        return x[0].numpy(), dict(cmap='gray', vmin=0., vmax=1.)
    return x.permute(1, 2, 0).numpy(), {}

@torch.no_grad()
def show(spark: SparK, x_bchw: torch.Tensor, active_b1ff: torch.BoolTensor = None, seed: int = 0, titles=None):
    x_bchw = x_bchw.to(DEVICE)
    if active_b1ff is None:     # random mask
        active_b1ff = spark.mask(x_bchw.shape[0], DEVICE, generator=torch.Generator().manual_seed(seed))
    inp, masked, rec_or_inp = spark(x_bchw, active_b1ff=active_b1ff, vis=True)
    loss = masked_loss(spark, inp, rec_or_inp, active_b1ff)
    err = (rec_or_inp - inp).abs().mean(dim=1, keepdim=True).clamp(0., 1.)    # 0 on visible patches
    
    B = x_bchw.shape[0]
    fig, axes = plt.subplots(B, 4, figsize=(13, 3.3 * B), squeeze=False)
    for r in range(B):
        for c, (name, bchw) in enumerate([('input', inp), ('masked input', masked), ('reconstructed', rec_or_inp)]):
            img, kw = to_img(bchw[r])
            axes[r][c].imshow(img, **kw)
            axes[r][c].set_title(name if r > 0 else f'{name}', fontsize=10)
        axes[r][3].imshow(err[r, 0].cpu().numpy(), cmap='magma', vmin=0., vmax=0.5)
        axes[r][3].set_title(f'|error| (loss={loss[r].item():.3f})', fontsize=10)
        if titles is not None:
            axes[r][0].set_ylabel(titles[r], fontsize=8)
        for a in axes[r]:
            a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()
    return loss.cpu()

## 5. 무작위 패치 여러 장으로 복원 확인

In [ ]:
idx = pick_patches(N_SHOW, SEED)
x = load_batch(idx)
print('indices:', idx, '| input range: [%.3f, %.3f], mean %.3f' % (x.min(), x.max(), x.mean()))
_ = show(spark, x, seed=SEED, titles=[os.path.basename(dataset.get_path(i))[-30:] for i in idx])

## 6. 같은 패치, 다른 마스크
마스크 위치가 달라져도 복원이 안정적인지 봅니다. 한 패치를 골라 마스크를 바꿔 가며 확인합니다.

In [ ]:
PATCH_NO = 0                        # 위 5번에서 본 패치 중 몇 번째를 쓸지
x1 = x[PATCH_NO:PATCH_NO + 1].repeat(4, 1, 1, 1)
act = torch.cat([spark.mask(1, DEVICE, generator=torch.Generator().manual_seed(100 + k)) for k in range(4)])
_ = show(spark, x1, active_b1ff=act)

## 7. 수치 확인: 학습 loss와 기준선 비교
`N_EVAL`개 패치에 무작위 마스크를 적용하고, 마스크된 영역의 loss를 비교합니다(모두 학습 loss와 같은 정규화).

| 항목 | 의미 |
|---|---|
| **SparK** | 이 모델의 loss |
| **평균값 기준선** | 가려진 패치를 평균 밝기로 채웠을 때. 정의상 약 1.0 |
| **블러 기준선(참고)** | 원본을 5×5 평균 필터로 흐린 영상을 정답으로 냈을 때. 원본을 *알고 있는* 값이라 모델이 낼 수 없는 수치이고, "가려진 영역의 노이즈(스페클) 몫이 얼마나 되는지"를 가늠하는 **대략적인** 참고치입니다. |

- SparK이 **1.0에 가까우면**: 가려진 영역의 구조를 거의 못 복원하는 상태입니다.
- SparK이 **블러 기준선에 가까우면**: loss가 더 내려갈 여지가 작습니다(남은 loss는 대부분 노이즈).
- 그 사이에 있으면: 구조 일부를 복원하고 있고, 더 학습하면 좋아질 여지가 있습니다.

In [ ]:
@torch.no_grad()
def evaluate(spark: SparK, n: int, seed: int, blur_k: int = 5, batch: int = 32):
    idx = pick_patches(n, seed + 1)
    res = {'SparK': [], 'mean baseline': [], 'blur oracle (ref.)': []}
    g = torch.Generator().manual_seed(seed + 2)
    for s in range(0, len(idx), batch):
        x = load_batch(idx[s:s + batch]).to(DEVICE)
        act = spark.mask(x.shape[0], DEVICE, generator=g)
        inp, _, rec_or_inp = spark(x, active_b1ff=act, vis=True)
        res['SparK'].append(masked_loss(spark, inp, rec_or_inp, act).cpu())
        mean = spark.unpatchify(spark.patchify(inp).mean(dim=-1, keepdim=True).expand(-1, -1, spark.patchify(inp).shape[-1]))
        res['mean baseline'].append(masked_loss(spark, inp, mean, act).cpu())
        blurred = F.avg_pool2d(F.pad(inp, (blur_k // 2,) * 4, mode='reflect'), blur_k, stride=1)
        res['blur oracle (ref.)'].append(masked_loss(spark, inp, blurred, act).cpu())
    return {k: torch.cat(v) for k, v in res.items()}

res = evaluate(spark, N_EVAL, SEED)
for k, v in res.items():
    print(f'{k:20s}: mean {v.mean():.4f}   std {v.std():.4f}   (n={len(v)})')

plt.figure(figsize=(6.5, 3.5))
plt.hist([v.numpy() for v in res.values()], bins=20, label=list(res.keys()))
plt.xlabel('masked-region loss per patch'); plt.ylabel('#patches'); plt.legend(); plt.tight_layout(); plt.show()

## 8. (선택) 여러 체크포인트 비교
epoch별로 복사해 둔 체크포인트가 있다면, 같은 패치·같은 마스크로 loss를 나란히 비교할 수 있습니다.

In [ ]:
CKPTS = {
    # 'epoch6': './exp_ckpts/resnet101vd_withdecoder_1kpretrained_spark_style_ep6.pth',
    # 'epoch12': './exp_ckpts/resnet101vd_withdecoder_1kpretrained_spark_style_ep12.pth',
}
for name, path in CKPTS.items():
    sp, sz = build_spark(path)
    assert sz == input_size, 'input_size differs'
    r = evaluate(sp, N_EVAL, SEED)
    print(f"{name:10s}: SparK {r['SparK'].mean():.4f} | mean baseline {r['mean baseline'].mean():.4f} | blur oracle {r['blur oracle (ref.)'].mean():.4f}")
    del sp